In [1]:
import pandas as pd
import numpy as np
import joblib
import sys
import os

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

for _mod in list(sys.modules.keys()):
    if _mod.startswith("src"):
        del sys.modules[_mod]
from src.data_loader import DataLoader

In [2]:
test_data_path = r"..\data\raw\test.csv"
load_the_data = DataLoader()
df = load_the_data.load_data(test_data_path)

pd.set_option('display.max_columns', None)
df.head()

,Id,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,1,4,2015-09-17,1.0,1,0,0,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,3,4,2015-09-17,1.0,1,0,0,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
2,3,7,4,2015-09-17,1.0,1,0,0,a,c,24000.0,4.0,2013.0,0,NaN,NaN,NaN
3,4,8,4,2015-09-17,1.0,1,0,0,a,a,7520.0,10.0,2014.0,0,NaN,NaN,NaN
4,5,9,4,2015-09-17,1.0,1,0,0,a,c,2030.0,8.0,2000.0,0,NaN,NaN,NaN


In [3]:
preprocessing_pipeline_path = r"..\models\preprocessing_pipeline.pkl"
preprocessing_pipeline = joblib.load(preprocessing_pipeline_path)

In [4]:
competition_step = preprocessing_pipeline.named_steps['competition_features']

print(hasattr(competition_step, 'competition_distance_mean_'))

True


In [5]:
id = df['Id']

preprocessed_df = preprocessing_pipeline.transform(df)
preprocessed_df.head()

[FillOpenMedian] filling 11 missing Open values with median 1.0
[CompetitionFeatureEngineer] filling 96 missing CompetitionDistance values with training mean


,StateHoliday,PromoInterval,StoreType,Assortment,DayOfWeek,Open,Promo,SchoolHoliday,CompetitionDistance,Promo2,Promo2SinceDays,CompetitionOpenSinceDays,Year,Month
0,0.0,3.0,2.0,0.0,4,1.0,1,0,1270.0,0,0.0,2572.0,2015,9
1,0.0,1.0,0.0,0.0,4,1.0,1,0,14130.0,1,1627.0,3212.0,2015,9
2,0.0,3.0,0.0,2.0,4,1.0,1,0,24000.0,0,0.0,899.0,2015,9
3,0.0,3.0,0.0,0.0,4,1.0,1,0,7520.0,0,0.0,351.0,2015,9
4,0.0,3.0,0.0,2.0,4,1.0,1,0,2030.0,0,0.0,5525.0,2015,9


In [ ]:
model_path = r"..\backend\models\xgboost_model.pkl"
model = joblib.load(model_path)

In [7]:
predicted_data = model.predict(preprocessed_df)

In [8]:
predicted_data

array([ 4835.3843,  7981.284 ,  9805.454 , ...,  5570.4297, 20240.363 ,
        6777.281 ], shape=(41088,), dtype=float32)

In [9]:
predicted_df = pd.DataFrame()

predicted_df['Id'] = id
predicted_df['Sales'] = predicted_data

predicted_df.head()

,Id,Sales
0,1,4835.384277
1,2,7981.284180
2,3,9805.454102
3,4,6324.828613
4,5,7665.134277


In [10]:
len(predicted_df), len(df)

(41088, 41088)

In [11]:
submission_path = r"..\data\predicted\sumission.csv"
predicted_df.to_csv(submission_path, index=False)